In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("final-dataset.csv", dtype=str)

chicken = df.iloc[1:, 3].tolist()
fetus = df.iloc[1:, 4].tolist()
states = df.iloc[1:, 5].tolist()
breastcancer = df.iloc[1:, 6].tolist()

In [2]:
import csv
import time
import random
import urllib.parse
from datetime import datetime

import tldextract
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ── Config ────────────────────────────────────────────────────────────────────

OUTPUT_CSV  = "google_breastcancer_results.csv"
MAX_RESULTS = 10
DELAY       = 5.0

# ── Load queries ──────────────────────────────────────────────────────────────

df = pd.read_csv("final-dataset.csv", dtype=str)
queries = [q.strip() for q in breastcancer if isinstance(q, str) and q.strip()]

print(f"Loaded {len(queries)} breastcancer queries:")
for i, q in enumerate(queries, 1):
    print(f"  {i}. {q!r}")

# ── Browser ───────────────────────────────────────────────────────────────────

def make_driver():
    opts = Options()
    opts.add_argument("--incognito")
    opts.add_argument("--no-first-run")
    opts.add_argument("--no-default-browser-check")
    opts.add_argument("--disable-extensions")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1280,900")
    opts.add_argument(
        "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
    service = Service(ChromeDriverManager().install())
    driver  = webdriver.Chrome(service=service, options=opts)
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"},
    )
    return driver

def build_url(query: str) -> str:
    params = {"q": query, "hl": "en", "gl": "us", "num": "10", "udm": "14", "pws": "0"}
    return "https://www.google.com/search?" + urllib.parse.urlencode(params)

def extract_domain(url: str) -> str:
    """Full netloc minus www, e.g. cooking.nytimes.com"""
    try:
        return urllib.parse.urlparse(url).netloc.replace("www.", "").strip()
    except Exception:
        return ""

def extract_registrable_domain(url: str) -> str:
    """eTLD+1 only, e.g. nytimes.com — handles co.uk, com.au etc correctly"""
    try:
        ext = tldextract.extract(url)
        return f"{ext.domain}.{ext.suffix}" if ext.suffix else ext.domain
    except Exception:
        return ""

# ── Scraper — updated for current Google DOM ─────────────────────────────────

def scrape_page(driver, max_results=10) -> list[dict]:
    """
    tF2Cxc  = current Google organic result container (confirmed in your DOM)
    VwiC3b  = snippet div (also confirmed in your DOM)
    """
    wait = WebDriverWait(driver, 10)
    try:
        # Wait for tF2Cxc — the confirmed current container class
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.tF2Cxc")))
    except Exception:
        print("  WARNING: timed out waiting for div.tF2Cxc")
        # Print what's on the page to help debug further
        print("  Page title:", driver.title)
        return []

    containers = driver.find_elements(By.CSS_SELECTOR, "div.tF2Cxc")
    print(f"  Found {len(containers)} result containers (div.tF2Cxc)")

    results   = []
    seen_urls = set()
    rank      = 0

    for container in containers:
        if rank >= max_results:
            break

        # ── Title ─────────────────────────────────────────────────────────
        title = ""
        for sel in ["h3", "h3.LC20lb"]:
            try:
                title = container.find_element(By.CSS_SELECTOR, sel).text.strip()
                if title:
                    break
            except Exception:
                continue
        if not title:
            continue

        # ── URL ───────────────────────────────────────────────────────────
        url = ""
        try:
            for a in container.find_elements(By.CSS_SELECTOR, "a[href]"):
                href = a.get_attribute("href") or ""
                if href.startswith("http") and "google.com" not in href:
                    url = href
                    break
        except Exception:
            pass
        if not url or url in seen_urls:
            continue
        seen_urls.add(url)

        domain              = extract_domain(url)
        registrable_domain  = extract_registrable_domain(url)

        # ── Snippet ───────────────────────────────────────────────────────
        snippet = ""
        for sel in [
            "div.VwiC3b",          # confirmed in your DOM
            "div.VwiC3b span",
            "span.aCOpRe",
            "div.IsZvec",
        ]:
            try:
                snippet = container.find_element(By.CSS_SELECTOR, sel).text.strip()
                if snippet:
                    break
            except Exception:
                continue

        rank += 1
        results.append({
            "rank":                rank,
            "title":               title,
            "url":                 url,
            "domain":              domain,
            "registrable_domain":  registrable_domain,
            "snippet":             snippet,
        })
        print(f"    [{rank:>2}] {registrable_domain:<30} {title[:50]}")

    return results

# ── CSV setup ─────────────────────────────────────────────────────────────────

fields = [
    "query_index", "col", "query",
    "rank", "title", "url", "domain", "registrable_domain", "snippet",
    "n_results", "timestamp",
]

# ── Main loop ─────────────────────────────────────────────────────────────────

driver = make_driver()

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()

    for i, query in enumerate(queries):
        print(f"\n{'='*60}")
        print(f"Query {i+1}/{len(queries)}: {query!r}")

        try:
            driver.get(build_url(query))
            time.sleep(random.uniform(2.5, 4.0))

            results   = scrape_page(driver, max_results=MAX_RESULTS)
            timestamp = datetime.utcnow().isoformat()

            if results:
                for r in results:
                    writer.writerow({
                        "query_index": i + 1,
                        "col":         "NPC",
                        "query":       query,
                        **r,
                        "n_results":   len(results),
                        "timestamp":   timestamp,
                    })
                print(f"  → {len(results)} results saved")
            else:
                writer.writerow({
                    "query_index": i+1, "col": "NPC", "query": query,
                    "rank": None, "title": "", "url": "", "domain": "",
                    "snippet": "", "n_results": 0, "timestamp": timestamp,
                })
                print("  → 0 results")

            f.flush()

        except Exception as e:
            print(f"  ERROR: {e}")
            writer.writerow({
                "query_index": i+1, "col": "NPC", "query": query,
                "rank": None, "title": f"ERROR: {e}", "url": "",
                "domain": "", "snippet": "", "n_results": 0,
                "timestamp": datetime.utcnow().isoformat(),
            })
            f.flush()

        if i < len(queries) - 1:
            sleep = DELAY + random.uniform(0.5, 2.0)
            print(f"  Sleeping {sleep:.1f}s...")
            time.sleep(sleep)

driver.quit()
print(f"\nDone. Results saved to: {OUTPUT_CSV}")

Loaded 227 breastcancer queries:
  1. 'breast cancer risk pre and post abortion'
  2. 'd'
  3. 'Is breast cancer risk higher after having an abortion'
  4. 'breast cancer related to abortion'
  5. 'abortion breast cancer'
  6. 'Increase in breast cancer after abortion'
  7. 'does the risk of breast cancer increase after an abortion'
  8. 'risk of breath cancer after having an abortion'
  9. 'does breast cancer have an increase on abortion?'
  10. 'z'
  11. 'does abortions increase great cancer risk'
  12. 'v'
  13. 'Does breast cancer risk increase after an abortion'
  14. 'abortion breast cancer'
  15. 'risk of breast cancer after abortion'
  16. 'does getting an abortion increase the chances of getting breast cancer'
  17. 'does breast cancer increase after abortion'
  18. 'does you risk of breast cancer increase after having an abortion'
  19. 'risk of breast cancer after pregnancy'
  20. 'does the risk of breast cancer increase after getting an abortion'
  21. 'risk of breast cance

In [ ]:
df_breastcancer = pd.read_csv("google_breastcancer_results.csv")
df_breastcancer